In [2]:
#!/usr/bin/env python3

import pandas as pd
import numpy as np

K = 32

MODELS = [
    "randomforest",
    "xgboost",
    "mlp"
]


def expected(a,b):

    return 1/(1+10**((b-a)/400))


def elo_update(elo_a,elo_b,score_a,score_b):

    exp_a = expected(elo_a,elo_b)
    exp_b = expected(elo_b,elo_a)

    if score_a > score_b:

        s_a = 1
        s_b = 0

    elif score_a < score_b:

        s_a = 0
        s_b = 1

    else:

        s_a = 0.5
        s_b = 0.5

    elo_a += K*(s_a-exp_a)
    elo_b += K*(s_b-exp_b)

    return elo_a,elo_b


def main():

    df = pd.read_csv("results/summary.csv")

    elo = {m:1000 for m in MODELS}

    grouped = df.groupby([
        "dataset",
        "detector",
        "treatment"
    ])

    matches = 0

    for _,g in grouped:

        scores = {}

        for _,row in g.iterrows():

            scores[row["model"]] = row["mean"]

        present = [
            m for m in MODELS
            if m in scores
        ]

        for i in range(len(present)):

            for j in range(i+1,len(present)):

                a = present[i]
                b = present[j]

                elo[a],elo[b] = elo_update(

                    elo[a],
                    elo[b],

                    scores[a],
                    scores[b]

                )

                matches += 1


    # normalize RF to 1000
    rf = elo["randomforest"]

    for m in elo:

        elo[m] = elo[m] - rf + 1000


    print("\nELO ratings")
    print("-----------")

    ranked = sorted(

        elo.items(),
        key=lambda x:x[1],
        reverse=True
    )

    for m,e in ranked:

        print(f"{m:15} {e:.1f}")


    print("\nTotal matches:",matches)


if __name__=="__main__":

    main()


ELO ratings
-----------
mlp             1375.0
xgboost         1214.9
randomforest    1000.0

Total matches: 420
